![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/llama.cpp/llama.cpp_in_Spark_NLP_AutoGGUFModel_Embeddings.ipynb)

# Import llama.cpp 🦙 models into Spark NLP 🚀

Let's keep in mind a few things before we start 😊

- Support for llama.cpp embeddings was introduced in `Spark NLP 5.5.1`, enabling quantized LLM inference on a wide range of devices. Please make sure you have upgraded to the latest Spark NLP release.
- You need to use your own `.gguf` model files, which also include the models from the [Hugging Face Models](https://huggingface.co/models?library=gguf).

## Download a GGUF Model

Lets download a GGUF model to test it out. For this, we will use [bartowski/Phi-3.5-mini-instruct-GGUF](https://huggingface.co/bartowski/Phi-3.5-mini-instruct-GGUF). It is a 3.8B parameter model which also is available in 4-bit quantization. 

We can download the model by selecting the q4 GGUF file from the "Files and versions" tab.

Once downloaded, we can directly import this model into Spark NLP!

In [ ]:
EXPORT_PATH = "Phi-3.5-mini-instruct-Q4_K_M.gguf"
! wget "https://huggingface.co/bartowski/Phi-3.5-mini-instruct-GGUF/resolve/main/Phi-3.5-mini-instruct-Q4_K_M.gguf?download=true" -O  {EXPORT_PATH}

## Import and Save AutGGUF models in Spark NLP

- Let's install and setup Spark NLP (if running it Google Colab)
- This part is pretty easy via our simple script

In [ ]:
# Only execute this if you are on Google Colab
! wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash

Let's start Spark with Spark NLP included via our simple `start()` function

In [ ]:
import sparknlp

# let's start Spark with Spark NLP with GPU enabled. If you don't have GPUs available remove this parameter.
spark = sparknlp.start(gpu=True)
print(sparknlp.version())

- Let's use the `loadSavedModel` functon in `AutoGGUFModel`
- Most params will be set automatically. They can also be set later after loading the model in `AutoGGUFModel` during runtime, so don't worry about setting them now.
- `loadSavedModel` accepts two params, first is the path to the exported model. The second is the SparkSession that is `spark` variable we previously started via `sparknlp.start()`
- We can set the model to embedding mode with `setEmbedding`. Afterwards the model will return the embeddings in the Annotations.
- NOTE: `loadSavedModel` accepts local paths in addition to distributed file systems such as `HDFS`, `S3`, `DBFS`, etc. This feature was introduced in Spark NLP 4.2.2 release. Keep in mind the best and recommended way to move/share/reuse Spark NLP models is to use `write.save` so you can use `.load()` from any file systems natively.

In [ ]:
from sparknlp.annotator import *

# All these params should be identical to the original ONNX model
autoGGUFModel = (
    AutoGGUFModel.loadSavedModel(EXPORT_PATH, spark)
    .setInputCols("document")
    .setOutputCol("embeddings")
    .setBatchSize(4)
    .setEmbedding(True)
    .setNGpuLayers(99)
)

jsl-llama: Extracted 'libjllama.so' to '/tmp/libjllama.so'


- Let's save it on disk so it is easier to be moved around and also be used later via `.load` function

In [ ]:
autoGGUFModel.write().overwrite().save(f"Phi-3.5-mini-instruct-Q4_K_M_spark_nlp")

Let's clean up stuff we don't need anymore

In [ ]:
!rm -rf {EXPORT_PATH}

Awesome  😎 !

This is your GGUF model from loaded and saved by Spark NLP 🚀

In [ ]:
! ls -l Phi-3.5-mini-instruct-Q4_K_M_spark_nlp

total 2337164
drwxr-xr-x 2 root root       4096 Oct 19 11:45 metadata
-rwxrwxr-x 1 root root 2393232672 Oct 19 11:45 Phi-3.5-mini-instruct-Q4_K_M.gguf


Now let's see how we can use it on other machines, clusters, or any place you wish to use your new and shiny GGUF model 😊

In [ ]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline

document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")

autoGGUFModel = AutoGGUFModel.load("Phi-3.5-mini-instruct-Q4_K_M_spark_nlp")

pipeline = Pipeline().setStages([document_assembler, autoGGUFModel])

data = spark.createDataFrame([["This is a sentence."]]).toDF("text")

result = pipeline.fit(data).transform(data)
result.select("embeddings.embeddings").show(truncate=False)

24/10/19 11:52:55 WARN DAGScheduler: Broadcasting large task binary with size 1042.2 KiB
24/10/19 11:52:57 WARN DAGScheduler: Broadcasting large task binary with size 1042.2 KiB
24/10/19 11:52:57 WARN DAGScheduler: Broadcasting large task binary with size 1042.2 KiB
llama_model_loader: loaded meta data with 40 key-value pairs and 197 tensors from /tmp/spark-0375a9c3-860e-4fd1-aff8-49597303b20f/userFiles-b664e666-88c2-4f1b-bf41-3ee678d08309/Phi-3.5-mini-instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = phi3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Phi 3.5 Mini Instruct
llama_model_loader: - kv   3:                           general.finetune str            

[WARN] Not compiled with GPU offload support, --n-gpu-layers option will be ignored. See main README.md for information on enabling GPU BLAS support n_gpu_layers=-1
[INFO] build info build=3534 commit="641f5dd2"
[INFO] system info n_threads=6 n_threads_batch=-1 total_threads=6 system_info="AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | AVX512_BF16 = 0 | FMA = 1 | NEON = 0 | SVE = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | MATMUL_INT8 = 0 | LLAMAFILE = 1 | "


llm_load_tensors:        CPU buffer size =  2281.66 MiB
............................................................................................
llama_new_context_with_model: n_ctx      = 4096
llama_new_context_with_model: n_batch    = 512
llama_new_context_with_model: n_ubatch   = 512
llama_new_context_with_model: flash_attn = 0
llama_new_context_with_model: freq_base  = 10000.0
llama_new_context_with_model: freq_scale = 1
llama_kv_cache_init:        CPU KV buffer size =  1536.00 MiB       (0 + 1) / 1]
llama_new_context_with_model: KV self size  = 1536.00 MiB, K (f16):  768.00 MiB, V (f16):  768.00 MiB
llama_new_context_with_model:        CPU  output buffer size =     0.06 MiB
llama_new_context_with_model:        CPU compute buffer size =   300.01 MiB
llama_new_context_with_model: graph nodes  = 1286
llama_new_context_with_model: graph splits = 1


[INFO] initializing slots n_slots=4
[INFO] new slot id_slot=0 n_ctx_slot=1024
[INFO] new slot id_slot=1 n_ctx_slot=1024
[INFO] new slot id_slot=2 n_ctx_slot=1024
[INFO] new slot id_slot=3 n_ctx_slot=1024
[INFO] model loaded
[INFO] chat template chat_example="<|system|>\nYou are a helpful assistant<|end|>\n<|user|>\nHello<|end|>\n<|assistant|>\nHi there<|end|>\n<|user|>\nHow are you?<|end|>\n<|assistant|>\n" built_in=true
[INFO] all slots are idle
[INFO] slot is processing task id_slot=0 id_task=0
[INFO] kv cache rm [p0, end) id_slot=0 id_task=0 p0=0
[INFO] slot released id_slot=0 id_task=0 n_ctx=4096 n_past=5 n_system_tokens=0 n_cache_tokens=0 truncated=false
[INFO] all slots are idle
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

That's it! You can now go wild and use hundreds of GGUF models from HuggingFace 🤗 in Spark NLP 🚀
